# Signal Validation — Research Protocol

Evaluación de 14 señales (6 técnicas + 8 fundamentales) sobre el universo watchlist.

**Pregunta central**: ¿cada señal rankea correctamente quién rinde más y quién menos?

**Métricas clave**:
- IC (Information Coefficient): correlación de ranking entre señal y retorno futuro
- Quintile spread: ¿el top quintil supera al bottom?
- Monotonicity: ¿los quintiles están ordenados?
- Stability: ¿funciona en bull Y bear markets?

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from src.data.database import MarketDB
from src.data.universe import load_universe
from src.signals import technical, fundamental
from src.validation.signal_tester import SignalTester

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Load data
uni = load_universe('../config/universe.yaml')
tickers = uni.all_tickers + ['SPY']

data = yf.download(tickers, start='2022-01-01', progress=False)
prices = data['Close'].dropna(axis=1, thresh=int(len(data)*0.7))
volumes = data['Volume'].reindex(columns=prices.columns)

print(f'Universe: {len(prices.columns)} tickers')
print(f'Period: {prices.index[0].date()} to {prices.index[-1].date()} ({len(prices)} trading days)')
print(f'Tickers: {list(prices.columns)}')

In [ ]:
# Compute all signals
tech_signals = technical.compute_all(prices, volumes)

db = MarketDB('../data/db/market.db')
us_tickers = [t for t in uni.tickers_us if t in prices.columns]
fund_scores = fundamental.compute_all(db, us_tickers)

# Convert fundamental dicts to DataFrames (static, broadcast across dates)
fund_signals = {}
for name, scores in fund_scores.items():
    s = pd.Series(scores)
    fund_signals[name] = pd.DataFrame(
        np.tile(s.values, (len(prices), 1)),
        index=prices.index, columns=s.index,
    )

all_signals = {**tech_signals, **fund_signals}
print(f'Total signals: {len(all_signals)} ({len(tech_signals)} technical + {len(fund_signals)} fundamental)')

## 1. IC Summary — All Signals at 21d and 63d

In [ ]:
tester = SignalTester()

# Evaluate all signals at both horizons
results = []
for name, sig in all_signals.items():
    for h in [21, 63]:
        report = tester.full_evaluation(name, sig, prices, horizon=h, benchmark_ticker='SPY')
        results.append({
            'signal': name,
            'horizon': f'{h}d',
            'ic_mean': report.ic_mean,
            'ic_tstat': report.ic_tstat,
            'spread_ann': report.spread_annualized,
            'monotonicity': f'{report.monotonicity}/{report.n_groups-1}',
            'turnover': report.turnover_monthly,
            'verdict': report.verdict,
            'ic_bull': report.ic_bull,
            'ic_bear': report.ic_bear,
        })

summary_df = pd.DataFrame(results)

# Display 21d results
print('═' * 80)
print('SIGNAL EVALUATION — 21 DAY HORIZON')
print('═' * 80)
df21 = summary_df[summary_df['horizon'] == '21d'].sort_values('ic_mean', ascending=False)
for _, row in df21.iterrows():
    icon = {'PASS': '✅', 'MARGINAL': '⚠️', 'FAIL': '❌'}[row['verdict']]
    print(f"{icon} {row['signal']:25s} IC={row['ic_mean']:+.4f}  t={row['ic_tstat']:+6.2f}  "
          f"spread={row['spread_ann']:+7.1%}  mono={row['monotonicity']}  TO={row['turnover']:.0%}")

print()
print('═' * 80)
print('SIGNAL EVALUATION — 63 DAY HORIZON')
print('═' * 80)
df63 = summary_df[summary_df['horizon'] == '63d'].sort_values('ic_mean', ascending=False)
for _, row in df63.iterrows():
    icon = {'PASS': '✅', 'MARGINAL': '⚠️', 'FAIL': '❌'}[row['verdict']]
    print(f"{icon} {row['signal']:25s} IC={row['ic_mean']:+.4f}  t={row['ic_tstat']:+6.2f}  "
          f"spread={row['spread_ann']:+7.1%}  mono={row['monotonicity']}")

## 2. IC Heatmap — Signals × Horizons

In [ ]:
# Pivot for heatmap
ic_pivot = summary_df.pivot(index='signal', columns='horizon', values='ic_mean')
ic_pivot = ic_pivot.sort_values('21d', ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(ic_pivot.values, cmap='RdYlGn', aspect='auto', vmin=-0.15, vmax=0.15)

ax.set_xticks(range(len(ic_pivot.columns)))
ax.set_xticklabels(ic_pivot.columns)
ax.set_yticks(range(len(ic_pivot.index)))
ax.set_yticklabels(ic_pivot.index)

# Add values
for i in range(len(ic_pivot.index)):
    for j in range(len(ic_pivot.columns)):
        val = ic_pivot.iloc[i, j]
        color = 'white' if abs(val) > 0.08 else 'black'
        ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, label='IC Mean')
ax.set_title('Information Coefficient by Signal and Horizon', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Rolling IC — Signals That Pass/Marginal

¿La señal funciona consistentemente o solo en ciertos periodos?

In [ ]:
# Select signals with IC > 0.03 at either horizon
passing = summary_df[
    (summary_df['ic_mean'] > 0.03) & (summary_df['horizon'] == '21d')
]['signal'].tolist()

fwd_ret_21 = tester.compute_forward_returns(prices, 21)

fig, axes = plt.subplots(len(passing), 1, figsize=(14, 4 * len(passing)), sharex=True)
if len(passing) == 1:
    axes = [axes]

for ax, name in zip(axes, passing):
    ic_series = tester.compute_ic(all_signals[name], fwd_ret_21)
    rolling_ic = ic_series.rolling(63, min_periods=21).mean()

    ax.fill_between(rolling_ic.index, 0, rolling_ic.values,
                    where=rolling_ic.values >= 0, alpha=0.3, color='green')
    ax.fill_between(rolling_ic.index, 0, rolling_ic.values,
                    where=rolling_ic.values < 0, alpha=0.3, color='red')
    ax.plot(rolling_ic.index, rolling_ic.values, color='black', linewidth=1)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0.03, color='green', linestyle=':', alpha=0.5, label='IC=0.03 threshold')

    overall_ic = ic_series.mean()
    ax.set_title(f'{name}  (overall IC = {overall_ic:+.4f})', fontsize=12, fontweight='bold')
    ax.set_ylabel('Rolling IC (63d)')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle('Rolling IC (63-day window) — Signals with IC > 0.03', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Quintile Analysis — Top Signals

¿El top quintil consistentemente supera al bottom?

In [ ]:
top_signals = ['revenue_growth', 'momentum_12_1', 'roe', 'debt_equity_inv', 'gross_margin_delta']
top_signals = [s for s in top_signals if s in all_signals]

fig, axes = plt.subplots(1, len(top_signals), figsize=(4 * len(top_signals), 5), sharey=True)
if len(top_signals) == 1:
    axes = [axes]

for ax, name in zip(axes, top_signals):
    sig = all_signals[name]
    report = tester.full_evaluation(name, sig, prices, horizon=21, benchmark_ticker='SPY')
    q_rets = report.quintile_returns
    ann_rets = [r * 252 / 21 for r in q_rets]  # annualize

    colors = ['#d32f2f' if r < 0 else '#388e3c' for r in ann_rets]
    bars = ax.bar(range(1, len(q_rets) + 1), [r * 100 for r in ann_rets], color=colors, alpha=0.8)
    ax.set_xlabel('Quintile')
    ax.set_xticks(range(1, len(q_rets) + 1))
    ax.set_xticklabels([f'Q{i}\n(worst)' if i == 1 else f'Q{i}\n(best)' if i == len(q_rets) else f'Q{i}'
                        for i in range(1, len(q_rets) + 1)])

    v = report.verdict
    icon = {'PASS': '✅', 'MARGINAL': '⚠️', 'FAIL': '❌'}[v]
    ax.set_title(f'{name}\n{icon} IC={report.ic_mean:+.3f}', fontsize=11, fontweight='bold')

    # Add value labels
    for bar, val in zip(bars, ann_rets):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.0f}%', ha='center', va='bottom', fontsize=9)

axes[0].set_ylabel('Annualized Return (%)')
plt.suptitle('Quintile Returns (21d horizon, annualized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Regime Analysis — Bull vs Bear

¿Las señales funcionan en ambos regímenes o solo en uno?

In [ ]:
# Regime data
regime_data = []
for name in top_signals:
    row_21 = summary_df[(summary_df['signal'] == name) & (summary_df['horizon'] == '21d')].iloc[0]
    regime_data.append({
        'signal': name,
        'IC Overall': row_21['ic_mean'],
        'IC Bull': row_21['ic_bull'] if row_21['ic_bull'] is not None else 0,
        'IC Bear': row_21['ic_bear'] if row_21['ic_bear'] is not None else 0,
    })

regime_df = pd.DataFrame(regime_data).set_index('signal')

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(regime_df))
w = 0.25

ax.bar(x - w, regime_df['IC Overall'], w, label='Overall', color='#1976d2', alpha=0.8)
ax.bar(x, regime_df['IC Bull'], w, label='Bull Market', color='#388e3c', alpha=0.8)
ax.bar(x + w, regime_df['IC Bear'], w, label='Bear Market', color='#d32f2f', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(regime_df.index, rotation=30, ha='right')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=0.03, color='green', linestyle=':', alpha=0.5, label='IC=0.03 threshold')
ax.set_ylabel('IC Mean')
ax.set_title('IC by Market Regime — Top Signals (21d)', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Current Signal Scores — Today's Snapshot

¿Cómo se ven las señales HOY para cada acción?

In [ ]:
# Get latest scores for passing signals
snapshot = {}

# Technical (latest date)
for name in ['momentum_12_1']:
    if name in tech_signals:
        latest = tech_signals[name].iloc[-1].dropna()
        for t in latest.index:
            if t not in snapshot:
                snapshot[t] = {}
            snapshot[t][name] = latest[t]

# Fundamental
for name in ['revenue_growth', 'roe', 'debt_equity_inv', 'gross_margin_delta']:
    if name in fund_scores:
        for t, v in fund_scores[name].items():
            if t not in snapshot:
                snapshot[t] = {}
            snapshot[t][name] = v

snap_df = pd.DataFrame(snapshot).T
snap_df['composite'] = snap_df.mean(axis=1)
snap_df = snap_df.sort_values('composite', ascending=False)

# Display as heatmap
fig, ax = plt.subplots(figsize=(12, 8))
display_cols = [c for c in snap_df.columns if c != 'composite']
display_df = snap_df[display_cols + ['composite']]

im = ax.imshow(display_df.values, cmap='RdYlGn', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(display_df.columns)))
ax.set_xticklabels(display_df.columns, rotation=45, ha='right')
ax.set_yticks(range(len(display_df.index)))
ax.set_yticklabels(display_df.index)

for i in range(len(display_df.index)):
    for j in range(len(display_df.columns)):
        val = display_df.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=10, color=color)

plt.colorbar(im, label='Score [-1, +1]')
ax.set_title('Current Signal Scores — Sorted by Composite (top = most bullish)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nRanking by composite score:')
for i, (ticker, row) in enumerate(snap_df.iterrows(), 1):
    print(f'  {i:>2}. {ticker:6s}  composite={row["composite"]:+.3f}')

## 7. IC by Year — Stability Over Time

In [ ]:
fwd_ret_21 = tester.compute_forward_returns(prices, 21)

yearly_ic = {}
for name in top_signals:
    ic_series = tester.compute_ic(all_signals[name], fwd_ret_21)
    for year, group in ic_series.groupby(ic_series.index.year):
        if len(group) >= 20:
            if name not in yearly_ic:
                yearly_ic[name] = {}
            yearly_ic[name][year] = group.mean()

yearly_df = pd.DataFrame(yearly_ic)

fig, ax = plt.subplots(figsize=(12, 6))
yearly_df.plot(kind='bar', ax=ax, alpha=0.8)
ax.axhline(y=0, color='gray', linestyle='--')
ax.axhline(y=0.03, color='green', linestyle=':', alpha=0.5, label='IC=0.03')
ax.set_ylabel('IC Mean')
ax.set_xlabel('Year')
ax.set_title('IC by Year — Top Signals (21d)', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- Signals that work in ALL years are most reliable')
print('- Signals that flip sign between years are regime-dependent')
print('- 2022 was a bear market — check which signals worked there')

## 8. Full Reports — Passing Signals

Reportes completos del Research Protocol para señales que pasan.

In [ ]:
for name in top_signals:
    report = tester.full_evaluation(name, all_signals[name], prices, horizon=21, benchmark_ticker='SPY')
    print(report.summary())
    print()

## 9. Caveats & Next Steps

**Limitaciones de este análisis:**
- **Universo muy pequeño (12-16 tickers)**: quintiles de 2-3 acciones son estadísticamente frágiles
- **Señales fundamentales son estáticas**: usamos el ratio más reciente para todas las fechas. En producción se actualizan trimestralmente
- **3 años de datos**: necesitamos más historia para conclusiones robustas
- **Sin factor attribution**: no hemos controlado por market/size/value/momentum

**Próximos pasos:**
1. Escalar universo a S&P 500 (cambiar `active_universe: sp500` en YAML)
2. Factor attribution: ¿el IC de revenue_growth es alpha real o solo growth-factor exposure?
3. Composite score: combinar señales que pasan → ranking unificado
4. Portfolio baseline: ¿el top quintil del composite supera SPY después de costos?